In [2]:
import pandas as pd
import numpy as np
import ast

In [3]:
credits = pd.read_csv('tmdb_5000_credits.csv')
movies = pd.read_csv('tmdb_5000_movies.csv')

In [4]:
movies = movies.merge(credits, on='title')

In [5]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [6]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [7]:
movies['genres'] = movies['genres'].apply(convert)

In [8]:
movies['genres']

0       [Action, Adventure, Fantasy, Science Fiction]
1                        [Adventure, Fantasy, Action]
2                          [Action, Adventure, Crime]
3                    [Action, Crime, Drama, Thriller]
4                [Action, Adventure, Science Fiction]
                            ...                      
4804                        [Action, Crime, Thriller]
4805                                [Comedy, Romance]
4806               [Comedy, Drama, Romance, TV Movie]
4807                                               []
4808                                    [Documentary]
Name: genres, Length: 4809, dtype: object

In [9]:
movies['keywords'] = movies['keywords'].apply(convert)

In [10]:
movies['keywords']

0       [culture clash, future, space war, space colon...
1       [ocean, drug abuse, exotic island, east india ...
2       [spy, based on novel, secret agent, sequel, mi...
3       [dc comics, crime fighter, terrorist, secret i...
4       [based on novel, mars, medallion, space travel...
                              ...                        
4804    [united states–mexico barrier, legs, arms, pap...
4805                                                   []
4806    [date, love at first sight, narration, investi...
4807                                                   []
4808            [obsession, camcorder, crush, dream girl]
Name: keywords, Length: 4809, dtype: object

In [11]:
movies['cast'] = movies['cast'].apply(lambda x: [i['name'] for i in ast.literal_eval(x)[:3]])  # Only top 3 actors

In [12]:
import ast

def safe_literal(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

In [13]:
movies['genres'] = movies['genres'].apply(safe_literal)
movies['keywords'] = movies['keywords'].apply(safe_literal)
movies['cast'] = movies['cast'].apply(safe_literal)
movies['crew'] = movies['crew'].apply(safe_literal)

In [14]:
print("safe_literal loaded")

safe_literal loaded


In [15]:
movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [16]:
def safe(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

In [17]:
movies['genres'] = movies['genres'].apply(safe)
movies['keywords'] = movies['keywords'].apply(safe)
movies['cast'] = movies['cast'].apply(safe)
movies['crew'] = movies['crew'].apply(safe)

In [18]:
movies['genres'] = movies['genres'].apply(lambda x: [i['name'] for i in x if isinstance(i, dict)])
movies['keywords'] = movies['keywords'].apply(lambda x: [i['name'] for i in x if isinstance(i, dict)])

movies['cast'] = movies['cast'].apply(lambda x: [i['name'] for i in x[:3] if isinstance(i, dict)])

movies['crew'] = movies['crew'].apply(lambda x: [i['name'] for i in x if isinstance(i, dict) and i.get('job')=='Director'])

In [19]:
movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [20]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))
movies['tags'] = movies['tags'].str.lower()

In [21]:
print(movies.columns)
print(movies[['title','tags']].head())

Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'tags'],
      dtype='str')
                                      title               tags
0                                    Avatar      james cameron
1  Pirates of the Caribbean: At World's End     gore verbinski
2                                   Spectre         sam mendes
3                     The Dark Knight Rises  christopher nolan
4                               John Carter     andrew stanton


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])

kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
movies['cluster'] = kmeans.fit_predict(tfidf_matrix)

In [23]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [24]:
def get_recommendations(title, cosine_sim=cosine_sim):

    if title not in movies['title'].values:
        return pd.DataFrame()

    idx = movies[movies['title'] == title].index[0]

    # Find selected movie's cluster
    cluster = movies.iloc[idx]["cluster"]

    # Get movies only from the same cluster
    cluster_movies = movies[movies["cluster"] == cluster]

    sim_scores = []

    for movie_idx in cluster_movies.index:
        sim_scores.append((movie_idx, cosine_sim[idx][movie_idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    movie_indices = [i[0] for i in sim_scores[1:11]]

    return movies[['title', 'movie_id']].iloc[movie_indices]

In [25]:
movies[['movie_id', 'title', 'cluster']]

,movie_id,title,cluster
0,19995,Avatar,0
1,285,Pirates of the Caribbean: At World's End,0
2,206647,Spectre,0
3,49026,The Dark Knight Rises,0
4,49529,John Carter,0
...,...,...,...
4804,9367,El Mariachi,0
4805,72766,Newlyweds,0
4806,231617,"Signed, Sealed, Delivered",0
4807,126186,Shanghai Calling,0


In [49]:
import pickle

# Save the FULL dataframe (including genres)
with open("movie_data.pkl", "wb") as file:
    pickle.dump((movies, cosine_sim), file)

print("Saved successfully!")

Saved successfully!


In [50]:
import pickle

with open("movie_data.pkl", "rb") as file:
    movies_check, cosine_sim = pickle.load(file)

print(movies_check.columns)
print(movies_check["genres"].head())

Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'tags', 'cluster'],
      dtype='str')
0    []
1    []
2    []
3    []
4    []
Name: genres, dtype: object


In [26]:
print(get_recommendations('The Dark Knight Rises'))

                title  movie_id
65    The Dark Knight       155
95       Interstellar    157336
96          Inception     27205
119     Batman Begins       272
1034         Insomnia       320
1197     The Prestige      1124
3576          Memento        77
2767               54      3682
2905         Triangle     26466
3111        Severance      5072


In [27]:
import pickle

with open("movie_data.pkl", "wb") as f:
    pickle.dump((movies, cosine_sim), f)

In [28]:
print(movies.columns)

Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'tags', 'cluster'],
      dtype='str')


In [29]:
user_history = pd.read_csv("user_history.csv")

In [30]:
import streamlit as st
import pandas as pd
import requests
import pickle

with open('movie_data.pkl', 'rb') as file:
    movies, cosine_sim = pickle.load(file)

user_history = pd.read_csv("user_history.csv")

In [42]:
with open("movie_data.pkl", "rb") as file:
    movies, cosine_sim = pickle.load(file)

In [46]:
print(movies["genres"].head())

0    []
1    []
2    []
3    []
4    []
Name: genres, dtype: object


In [32]:
user_history = pd.read_csv("user_history.csv")

In [33]:
movies[['movie_id', 'title', 'cluster']]

,movie_id,title,cluster
0,19995,Avatar,0
1,285,Pirates of the Caribbean: At World's End,0
2,206647,Spectre,0
3,49026,The Dark Knight Rises,0
4,49529,John Carter,0
...,...,...,...
4804,9367,El Mariachi,0
4805,72766,Newlyweds,0
4806,231617,"Signed, Sealed, Delivered",0
4807,126186,Shanghai Calling,0


In [34]:
movies[['title', 'cluster']].sort_values('cluster')

,title,cluster
0,Avatar,0
3113,Welcome to the Rileys,0
3114,Police Academy: Mission to Moscow,0
3115,Blood Done Sign My Name,0
3116,Cinco de Mayo: La Batalla,0
...,...,...
604,Osmosis Jones,7
296,End of Days,7
970,Hannibal Rising,7
98,The Hobbit: An Unexpected Journey,7


In [35]:
movies[movies['cluster'] == 0][['title', 'cluster']]

,title,cluster
0,Avatar,0
1,Pirates of the Caribbean: At World's End,0
2,Spectre,0
3,The Dark Knight Rises,0
4,John Carter,0
...,...,...
4804,El Mariachi,0
4805,Newlyweds,0
4806,"Signed, Sealed, Delivered",0
4807,Shanghai Calling,0


In [36]:
movies['cluster'].value_counts().sort_index()

cluster
0    4170
1      77
2      56
3      42
4     182
5     123
6      69
7      90
Name: count, dtype: int64

In [37]:
cluster_table = movies[['title', 'cluster']].head(20)
cluster_table

,title,cluster
0,Avatar,0
1,Pirates of the Caribbean: At World's End,0
2,Spectre,0
3,The Dark Knight Rises,0
4,John Carter,0
5,Spider-Man 3,0
6,Tangled,0
7,Avengers: Age of Ultron,0
8,Harry Potter and the Half-Blood Prince,0
9,Batman v Superman: Dawn of Justice,0


In [38]:
print(movies.columns.tolist())

['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'tags', 'cluster']


In [40]:
import pickle

with open("movie_data.pkl", "rb") as file:
    data = pickle.load(file)

print(type(data))
print(len(data))

<class 'tuple'>
2


In [41]:
print(data[0].columns)

Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'tags', 'cluster'],
      dtype='str')


In [43]:
st.write("Columns:", movies.columns.tolist())

2026-07-01 15:58:21.963 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 15:58:22.892 
  command:

    streamlit run C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-07-01 15:58:22.896 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 15:58:22.897 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 15:58:22.899 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 15:58:22.900 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 15:58:22.902 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 15:58:22.907 Thread 

In [44]:
st.write(movies["genres"].head())

2026-07-01 16:07:22.041 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 16:07:22.042 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 16:07:22.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [47]:
import pickle

with open("movie_data.pkl", "wb") as file:
    pickle.dump((movies, cosine_sim), file)

print("movie_data.pkl saved successfully!")

movie_data.pkl saved successfully!


In [48]:
import pickle

with open("movie_data.pkl", "rb") as file:
    movies_test, cosine_sim = pickle.load(file)

print(movies_test["genres"].head())

0    []
1    []
2    []
3    []
4    []
Name: genres, dtype: object
